In [ ]:
# Load in python libraries
import xarray as xr
import xrft as xf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import pandas as pd
import matplotlib.colors as colors

In [ ]:
# Here are the parameters for the specific case this calculation is done for
theta = 0.1  # slope angle
f = 1e-4  # coriolis frequency
Sinf = 1.0  # slope Burger number
Ri_inv = 0.95
N2 = Sinf**2*f**2/np.tan(theta)**2 # stratification
delta = 0.8  # wave magnitude
Vinf = 0.05  # interior velocity
gamma = (1+(1-Ri_inv**(-1))*Sinf**2)*(1+Sinf**2)**(-1)  # stratification parameter
lmbd = N2*gamma*np.tan(theta)/(f*np.cos(theta))  # geostrophic shear
h = Vinf/lmbd  # boundary layer height
fstar = f*np.cos(theta)*(1+Sinf**2)**(0.5)  # modified inertial frequency
beta = (1+Sinf**2)**(0.5)  # beta (portion of modified inertial frequency that modifies it)

In [ ]:
# # file = "flow_fields_height_130.4272905213473_interior_velocity_0.1_visc_1.0e-5_Sinf_0.2500256810990314_gamma_0.9697212409556922_theta_0.453_f_0.0001_N2_1.0e-5.nc"
file = "flow_fields_Sinf_1.0_Ri_inv_0.95_delta_0.8_drag_w_DFLUX.nc"
ocean_data=xr.open_dataset("/glade/derecho/scratch/knudsenl/data/new_data/final_draft/"+file).chunk(time=1)#.isel(time=slice(0,4000)) #i=103
seconds = ocean_data.time/pd.Timedelta("1s")
ocean_data = ocean_data.assign_coords(time = ('time',seconds.data))

In [ ]:
# # file = "flow_fields_height_130.4272905213473_interior_velocity_0.1_visc_1.0e-5_Sinf_0.2500256810990314_gamma_0.9697212409556922_theta_0.453_f_0.0001_N2_1.0e-5.nc"
file = "flow_fields_Sinf_1.0_Ri_inv_0.75_delta_0.8_drag_w_DFLUX.nc"
ocean_data_1 = xr.open_dataset("/glade/derecho/scratch/knudsenl/data/new_data/final_draft/"+file).chunk(time=1)#.isel(time=slice(0,4000)) #i=103
seconds1 = ocean_data_1.time/pd.Timedelta("1s")
ocean_data_1 = ocean_data_1.assign_coords(time = ('time',seconds1.data))

In [ ]:
upper_limit = -1
isopycnal_numbers = 15
t1 = 300
t2 = 310
t3 = 400
t4 = 410
v_max = 1e-4
lin_thresh = 1e-7
c_map = "RdYlBu"
fig, axs = plt.subplots(2, 2,sharex=True,sharey=False,figsize=(8,4),layout="constrained")
ocean_data.u.isel(time=t1).plot(ax=axs[0,0],label=" ",add_colorbar=False,cmap=c_map,norm=colors.SymLogNorm(linthresh=lin_thresh,vmin=-v_max,vmax=v_max),rasterized=True) # cmap="Accent"zC=slice(0,70)
(ocean_data.B+ocean_data.ba).isel(time=t1,yC=0).plot.contour(ax=axs[0,0],colors="k",levels=isopycnal_numbers,alpha=0.75)
axs[0,0].set(title=r"T="+str(round(ocean_data.time[t1].values*fstar/(2*np.pi),2)),xlabel=" ",ylabel="z [$m$]")
ocean_data.u.isel(time=t2).plot(ax=axs[0,1],label=" ",add_colorbar=False,cmap=c_map,norm=colors.SymLogNorm(linthresh=lin_thresh,vmin=-v_max,vmax=v_max),rasterized=True) # cmap="Accent"zC=slice(0,70)
(ocean_data.B+ocean_data.ba).isel(time=t2,yC=0).plot.contour(ax=axs[0,1],colors="k",levels=isopycnal_numbers,alpha=0.75)
axs[0,1].set(title=r"T="+str(round(ocean_data.time[t2].values*fstar/(2*np.pi),2)),xlabel=" ",ylabel=" ")
ocean_data.u.isel(time=t3).plot(ax=axs[1,0],label=" ",add_colorbar=False,cmap=c_map,norm=colors.SymLogNorm(linthresh=lin_thresh,vmin=-v_max,vmax=v_max),rasterized=True) # cmap="Accent"zC=slice(0,70)
(ocean_data.B+ocean_data.ba).isel(time=t3,yC=0).plot.contour(ax=axs[1,0],colors="k",levels=isopycnal_numbers,alpha=0.75)
axs[1,0].set(title=r"T="+str(round(ocean_data.time[t3].values*fstar/(2*np.pi),2)),xlabel="x [$m$]",ylabel="z [$m$]")
im = ocean_data.u.isel(time=t4).plot(ax=axs[1,1], label=" ",add_colorbar=False,cmap=c_map,norm=colors.SymLogNorm(linthresh=lin_thresh,vmin=-v_max,vmax=v_max),rasterized=True)
(ocean_data.B+ocean_data.ba).isel(time=t4,yC=0).plot.contour(ax=axs[1,1],colors="k",levels=isopycnal_numbers,alpha=0.75)
axs[1,1].set(title=r"T="+str(round(ocean_data.time[t4].values*fstar/(2*np.pi),2)),xlabel="x [$m$]",ylabel=" ")
fig.colorbar(im, ax=axs, label="u-velocity [$m\,s^{-1}$]",ticks=[1e-2,-1e-2,1e-3,-1e-3,1e-4,-1e-4,1e-5,-1e-5,1e-6,-1e-6,0]) 
fig.text(0.04,0.975, "(a.)")
fig.text(0.04,0.48, "(b.)")
plt.savefig("uvelocity_drag.png",dpi=300)
plt.show()

In [ ]:
def z_change(t, z, x):  # advection by isopycnals
    z_osc = delta*lmbd/fstar*np.sin(fstar*t+np.pi/2)*(h-z)
    return z_osc

In [ ]:
# Caclulate the modified inertial period
inertial_period = ocean_data.time*fstar/(2*np.pi)
ocean_data_ip = ocean_data.assign_coords(intp = ('time',inertial_period.data))

In [ ]:
# Calculate the Quasi-Lagrangian v-velocity
ti = 20
tf = 300
zi = 64
xi = 256
v_lagrange = ocean_data.v.sel(xC=ocean_data.xC[xi]+z_change(ocean_data.time[ti:tf], ocean_data.zC[zi], ocean_data.xC[xi]), 
                              zC=ocean_data.zC[zi], time=ocean_data.time[ti:tf], method="nearest")
# Drops the dimensions so that time series can be plotted
v_lagrange = v_lagrange.drop_vars(['xC','yF','zC'])

In [ ]:
# Calculate the Quasi-Lagrangian v-velocity
ti = 20
tf = 300
zi = 64
xi = 256
u_lagrange = ocean_data.u.sel(xF=ocean_data.xF[xi]+z_change(ocean_data.time[ti:tf], ocean_data.zC[zi], ocean_data.xF[xi]), 
                              zC=ocean_data.zC[zi], time=ocean_data.time[ti:tf], method="nearest")
# Drops the dimensions so that time series can be plotted
u_lagrange = u_lagrange.drop_vars(['xF','yC','zC'])

In [ ]:
# Plot the total power spectrum
transform_lagrange = xf.power_spectrum(u_lagrange,dim="time")
transform_lagrange.plot()
plt.xscale("symlog",linthresh=1e-10)

In [ ]:
upper_limit = -1
isopycnal_numbers = 15
t1 = 300
t2 = 310
t3 = 400
t4 = 410
v_max = 1e-4
lin_thresh = 1e-7
c_map = "RdYlBu"
fig, axs = plt.subplots(2, 2,sharex=True,sharey=False,figsize=(8,4),layout="constrained")
ocean_data.u.isel(time=t1).plot(ax=axs[0,0],label=" ",add_colorbar=False,cmap=c_map,norm=colors.SymLogNorm(linthresh=lin_thresh,vmin=-v_max,vmax=v_max),rasterized=True) # cmap="Accent"zC=slice(0,70)
(ocean_data.B+ocean_data.ba).isel(time=t1,yC=0).plot.contour(ax=axs[0,0],colors="k",levels=isopycnal_numbers,alpha=0.75)
axs[0,0].set(title=r"T="+str(round(ocean_data.time[t1].values*fstar/(2*np.pi),2)),xlabel=" ",ylabel="z [$m$]")
ocean_data.u.isel(time=t2).plot(ax=axs[0,1],label=" ",add_colorbar=False,cmap=c_map,norm=colors.SymLogNorm(linthresh=lin_thresh,vmin=-v_max,vmax=v_max),rasterized=True) # cmap="Accent"zC=slice(0,70)
(ocean_data.B+ocean_data.ba).isel(time=t2,yC=0).plot.contour(ax=axs[0,1],colors="k",levels=isopycnal_numbers,alpha=0.75)
axs[0,1].set(title=r"T="+str(round(ocean_data.time[t2].values*fstar/(2*np.pi),2)),xlabel=" ",ylabel=" ")
ocean_data_1.u.isel(time=t3).plot(ax=axs[1,0],label=" ",add_colorbar=False,cmap=c_map,norm=colors.SymLogNorm(linthresh=lin_thresh,vmin=-v_max,vmax=v_max),rasterized=True) # cmap="Accent"zC=slice(0,70)
(ocean_data_1.B+ocean_data_1.ba).isel(time=t3,yC=0).plot.contour(ax=axs[1,0],colors="k",levels=isopycnal_numbers,alpha=0.75)
axs[1,0].set(title=r"T="+str(round(ocean_data_1.time[t3].values*fstar/(2*np.pi),2)),xlabel="x [$m$]",ylabel="z [$m$]")
im = ocean_data_1.u.isel(time=t4).plot(ax=axs[1,1], label=" ",add_colorbar=False,cmap=c_map,norm=colors.SymLogNorm(linthresh=lin_thresh,vmin=-v_max,vmax=v_max),rasterized=True)
(ocean_data_1.B+ocean_data_1.ba).isel(time=t4,yC=0).plot.contour(ax=axs[1,1],colors="k",levels=isopycnal_numbers,alpha=0.75)
axs[1,1].set(title=r"T="+str(round(ocean_data_1.time[t4].values*fstar/(2*np.pi),2)),xlabel="x [$m$]",ylabel=" ")
fig.colorbar(im, ax=axs, label="u-velocity [$m\,s^{-1}$]",ticks=[1e-2,-1e-2,1e-3,-1e-3,1e-4,-1e-4,1e-5,-1e-5,1e-6,-1e-6,0]) 
fig.text(0.04,0.975, "(a.)")
fig.text(0.04,0.48, "(b.)")
plt.savefig("uvelocity_drag.png",dpi=300)
plt.show()

In [ ]:
upper_limit = -1
isopycnal_numbers = 15
t1 = 200
t2 = 210
t3 = 100
t4 = 110
v_max = 1e-6
lin_thresh = 1e-12
c_map = "RdYlBu"
fig, axs = plt.subplots(2, 2,sharex=True,sharey=False,figsize=(8,4),layout="constrained")
ocean_data.PV.isel(time=t1).plot(ax=axs[0,0],label=" ",add_colorbar=False,cmap=c_map,norm=colors.SymLogNorm(linthresh=lin_thresh,vmin=-v_max,vmax=v_max),rasterized=True) # cmap="Accent"zC=slice(0,70)
(ocean_data.B+ocean_data.ba).isel(time=t1,yC=0).plot.contour(ax=axs[0,0],colors="k",levels=isopycnal_numbers,alpha=0.75)
axs[0,0].set(title=r"T="+str(round(ocean_data.time[t1].values*fstar/(2*np.pi),2)),xlabel=" ",ylabel="z [$m$]")
ocean_data.PV.isel(time=t2).plot(ax=axs[0,1],label=" ",add_colorbar=False,cmap=c_map,norm=colors.SymLogNorm(linthresh=lin_thresh,vmin=-v_max,vmax=v_max),rasterized=True) # cmap="Accent"zC=slice(0,70)
(ocean_data.B+ocean_data.ba).isel(time=t2,yC=0).plot.contour(ax=axs[0,1],colors="k",levels=isopycnal_numbers,alpha=0.75)
axs[0,1].set(title=r"T="+str(round(ocean_data.time[t2].values*fstar/(2*np.pi),2)),xlabel=" ",ylabel=" ")
ocean_data_1.PV.isel(time=t3).plot(ax=axs[1,0],label=" ",add_colorbar=False,cmap=c_map,norm=colors.SymLogNorm(linthresh=lin_thresh,vmin=-v_max,vmax=v_max),rasterized=True) # cmap="Accent"zC=slice(0,70)
(ocean_data_1.B+ocean_data_1.ba).isel(time=t3,yC=0).plot.contour(ax=axs[1,0],colors="k",levels=isopycnal_numbers,alpha=0.75)
axs[1,0].set(title=r"T="+str(round(ocean_data_1.time[t3].values*fstar/(2*np.pi),2)),xlabel="x [$m$]",ylabel="z [$m$]")
im = ocean_data_1.PV.isel(time=t4).plot(ax=axs[1,1], label=" ",add_colorbar=False,cmap=c_map,norm=colors.SymLogNorm(linthresh=lin_thresh,vmin=-v_max,vmax=v_max),rasterized=True)
(ocean_data_1.B+ocean_data_1.ba).isel(time=t4,yC=0).plot.contour(ax=axs[1,1],colors="k",levels=isopycnal_numbers,alpha=0.75)
axs[1,1].set(title=r"T="+str(round(ocean_data_1.time[t4].values*fstar/(2*np.pi),2)),xlabel="x [$m$]",ylabel=" ")
fig.colorbar(im, ax=axs, label="u-velocity [$m\,s^{-1}$]",ticks=[1e-2,-1e-2,1e-3,-1e-3,1e-4,-1e-4,1e-5,-1e-5,1e-6,-1e-6,0]) 
fig.text(0.04,0.975, "(a.)")
fig.text(0.04,0.48, "(b.)")
plt.savefig("uvelocity_drag.png",dpi=300)
plt.show()

In [ ]:
# Calculate the Quasi-Lagrangian v-velocity
ti = 20
tf = 300
zi = 64
xi = 256
v_lagrange = ocean_data.v.sel(xC=ocean_data.xC[xi]+z_change(ocean_data.time[ti:tf], ocean_data.zC[zi], ocean_data.xC[xi]), 
                              zC=ocean_data.zC[zi], time=ocean_data.time[ti:tf], method="nearest")
# Drops the dimensions so that time series can be plotted
v_lagrange = v_lagrange.drop_vars(['xC','yF','zC'])

In [ ]:
# Calculate the Quasi-Lagrangian v-velocity
ti = 20
tf = 300
zi = 64
xi = 256
u_lagrange = ocean_data.u.sel(xF=ocean_data.xF[xi]+z_change(ocean_data.time[ti:tf], ocean_data.zC[zi], ocean_data.xF[xi]), 
                              zC=ocean_data.zC[zi], time=ocean_data.time[ti:tf], method="nearest")
# Drops the dimensions so that time series can be plotted
u_lagrange = u_lagrange.drop_vars(['xF','yC','zC'])

In [ ]:
transform_lagrange = xf.power_spectrum(u_lagrange,dim="time")


In [ ]:
upper_limit = -1
isopycnal_numbers = 15
t1 = 100
t2 = 250
v_max = 1e-2
lin_thresh = 1e-5
c_map = "bwr"


fig = plt.figure(figsize=(8, 6), layout='constrained')
axs = fig.subplot_mosaic([["time1","time2"],
                          ["frequency","frequency"]])

ocean_data.u.isel(time=t1).plot(ax=axs["time1"],label=" ",add_colorbar=False,cmap=c_map,norm=colors.SymLogNorm(linthresh=lin_thresh,vmin=-v_max,vmax=v_max),rasterized=True) # cmap="Accent"zC=slice(0,70)
(ocean_data.B+ocean_data.ba).isel(time=t1,yC=0).plot.contour(ax=axs["time1"],colors="k",levels=isopycnal_numbers,alpha=0.75)
axs["time1"].set(title=r"T="+str(round(ocean_data.time[t1].values*fstar/(2*np.pi),2)),xlabel="x [$m$]",ylabel="z [$m$]")
im = ocean_data.u.isel(time=t2).plot(ax=axs["time2"], add_colorbar=False, cmap=c_map, norm=colors.SymLogNorm(linthresh=lin_thresh, vmin=-v_max, vmax=v_max), rasterized=True)
fig.colorbar(im, ax=[axs["time1"], axs["time2"]], label="u-velocity [$m\\,s^{-1}$]",ticks=[1e-2,-1e-2,1e-3,-1e-3,1e-4,-1e-4,1e-5,-1e-5,0])
(ocean_data.B+ocean_data.ba).isel(time=t2,yC=0).plot.contour(ax=axs["time2"],colors="k",levels=isopycnal_numbers,alpha=0.75)
axs["time2"].set(title=r"T="+str(round(ocean_data.time[t2].values*fstar/(2*np.pi),2)),xlabel="x [$m$]",ylabel=" ")

axs["frequency"].plot(transform_lagrange.freq_time[:]*2*np.pi,transform_lagrange[:],color="k")
axs["frequency"].vlines(0.5*fstar,ymin=0,ymax=np.max(transform_lagrange)*1.05,color="b",alpha=0.5,linestyles="dotted",label="$f^*/2$")
axs["frequency"].vlines(fstar,ymin=0,ymax=np.max(transform_lagrange)*1.05,color="k",alpha=0.5,linestyles="dotted",label="$f^*$")
axs["frequency"].set_xlim([1e-05,transform_lagrange.freq_time[:].max()*2*np.pi])
axs["frequency"].set_ylim([0,np.max(transform_lagrange)*1.05])
axs["frequency"].set_xlabel("Frequency [s$^{-1}$]")
axs["frequency"].set_xscale("log")
axs["frequency"].set_ylabel("Power Spectrum Density [m$^3$ s$^{-2}$]")

fig.text(0.05,0.975, "(a.)")
fig.text(0.05,0.48, "(b.)")

plt.savefig("drag_figure.png",dpi=300)

plt.show()
